# Check after annotation

Sprawdzamy, ile mamy anotacji, jak się rozkładają, czy są jakieś anomalie. Otrzymujemy finalny zbiór dla modelu

In [54]:
import pandas as pd

sampled_df = pd.read_parquet("../data_all/above_value_minus1/sample_200k.parquet")

In [55]:
sampled_df

,uid,url,score,pos_keys,neg_keys,score_bin
0,54093d4356b5965572633d43ad9528d1,https://static1.squarespace.com/static/5a2d8c1...,-0.5,,city,bin_1
1,048409ec0da2ff018dd0abf30438ca3d,https://u.realgeeks.media/northernutahmls/roun...,-0.5,,city,bin_1
2,770466b5e608fb156d842fdb1b45bccc,https://assets.weather-forecast.com/maps/stati...,-0.5,,precipitation,bin_1
3,69674f3bfa313604ac2e2835af7d7886,http://us.123rf.com/450wm/peterhermesfurian/pe...,-0.5,,ciudades,bin_1
4,5de7faa4a10ff297560c6c315982a185,https://assets.weather-forecast.com/maps/stati...,-0.5,,precipitation,bin_1
...,...,...,...,...,...,...
199995,67f55022993ccb42a8604665a038820e,https://maps.googleapis.com/maps/api/staticmap...,0.0,,,bin_2
199996,768e30dd5e9007e0976ebae2244fe50f,https://m20.targeo.pl/i/cache/static/budynek/n...,0.0,,,bin_2
199997,7083d42b95e450dbacb687ea71d61834,http://ift.tt/1vhCEUd,0.0,,,bin_2
199998,2b83053953463eea3eafb9879bde19e4,https://backnumber.dailyportalz.jp/2015/05/13/...,0.0,,,bin_2


In [56]:
import json

with open("../data_all/above_value_minus1/annotations_progress.jsonl", "r") as f:
    data = [json.loads(line) for line in f]
jsonl_df = pd.DataFrame(data)
merged_df = pd.merge(sampled_df, jsonl_df, on="uid", how="inner")

In [57]:
annotation_count = merged_df['label'].notna().sum()
print(f"Number of rows with annotation: {annotation_count}")

Number of rows with annotation: 4829


In [58]:
valid_annotation_count = merged_df[merged_df['label'] != 'INVALID']['label'].notna().sum()
print(f"Number of rows with valid annotation (YES or NO): {valid_annotation_count}")

Number of rows with valid annotation (YES or NO): 3000


In [59]:
label_distribution = merged_df['label'].value_counts()
print("Label distribution:")
label_distribution

Label distribution:


label
NO         2932
INVALID    1829
YES          68
Name: count, dtype: int64

In [60]:
bin_labels = {
    'bin_1': '-0.5',
    'bin_2': '0.0',
    'bin_3': '0.5',
    'bin_4': '1.0',
    'bin_5': '1.5-2.0',
    'bin_6': '2.5-5.0',
    'bin_7': '5.5-10.0',
    'bin_8': '10.5+',
    'bin_9': '-1.0'
}

merged_df['score_bin'] = merged_df['score_bin'].map(bin_labels)
merged_df

,uid,url,score,pos_keys,neg_keys,score_bin,label,timestamp
0,47d936b9a2b73729f100922aaea4754b,https://assets.weather-forecast.com/maps/stati...,-0.5,,precipitation,-0.5,NO,2026-02-11T22:19:36.831885
1,eb7ed790974fe9549d8ec73ef1b5a40f,http://actualidadhumanitaria.com/wp-content/up...,-0.5,,miles,-0.5,NO,2026-02-11T22:28:48.142193
2,026ad7cea92db1bc23e34058544ee81b,https://i.hizliresim.com/6yAr9v.jpg,-0.5,,city,-0.5,INVALID,2026-02-11T22:14:27.394738
3,2deac59fba180138c78559a999b9654a,https://static1.millenium.org/article_old/imag...,-0.5,,ville,-0.5,NO,2026-02-11T22:07:26.311041
4,e72246916a1d6af335ef2d588e965282,https://volcanodiscovery.de/maps/quakemap-6127...,-0.5,,km,-0.5,NO,2026-02-11T22:23:11.915965
...,...,...,...,...,...,...,...,...
4824,2aa6fa2895707b9c79714ad1bb1c2bfa,https://images-na.ssl-images-amazon.com/images...,0.0,,,0.0,NO,2026-02-12T09:41:23.197836
4825,107cfacc8e1ac5d5459db21a56795450,https://www.simshack.net/images/fsdg-mauritius...,0.0,,,0.0,NO,2026-02-13T16:03:40.994234
4826,74d71221c5e626230109ecf471cc934c,http://news.southcn.com/content/images/attache...,0.0,,,0.0,INVALID,2026-02-13T00:08:37.651759
4827,6b066991600f05dd71752fda3ec12484,http://journalistsresource.org/wp-content/uplo...,1.0,tax,,1.0,NO,2026-02-11T22:30:44.935920


In [61]:
bin_stats = merged_df.groupby('score_bin').agg(
    yes_count=('label', lambda x: (x == 'YES').sum()),
    no_count=('label', lambda x: (x == 'NO').sum()),
    invalid_count=('label', lambda x: (x == 'INVALID').sum()),
    total_annotated=('label', 'count')
).reset_index()
print("Bin statistics:")
bin_stats

Bin statistics:


,score_bin,yes_count,no_count,invalid_count,total_annotated
0,-0.5,1,90,37,128
1,-1.0,2,118,43,163
2,0.0,31,2438,1549,4018
3,0.5,3,66,32,101
4,1.0,8,136,100,244
5,1.5-2.0,7,47,34,88
6,10.5+,2,6,4,12
7,2.5-5.0,9,24,27,60
8,5.5-10.0,5,7,3,15


In [62]:
bin_stats['yes_percent'] = (bin_stats['yes_count'] / bin_stats['total_annotated']) * 100
bin_stats['no_percent'] = (bin_stats['no_count'] / bin_stats['total_annotated']) * 100
bin_stats['invalid_percent'] = (bin_stats['invalid_count'] / bin_stats['total_annotated']) * 100
print("Bin statistics with percentages:")
bin_stats

Bin statistics with percentages:


,score_bin,yes_count,no_count,invalid_count,total_annotated,yes_percent,no_percent,invalid_percent
0,-0.5,1,90,37,128,0.781250,70.312500,28.906250
1,-1.0,2,118,43,163,1.226994,72.392638,26.380368
2,0.0,31,2438,1549,4018,0.771528,60.676954,38.551518
3,0.5,3,66,32,101,2.970297,65.346535,31.683168
4,1.0,8,136,100,244,3.278689,55.737705,40.983607
5,1.5-2.0,7,47,34,88,7.954545,53.409091,38.636364
6,10.5+,2,6,4,12,16.666667,50.000000,33.333333
7,2.5-5.0,9,24,27,60,15.000000,40.000000,45.000000
8,5.5-10.0,5,7,3,15,33.333333,46.666667,20.000000


# Sampling

| bin_label | score range | komentarz            |
| --------- | ----------- | -------------------- |
| bin_1     | -1 to -0.5  | bardzo niskie        |
| bin_2     | -0.5 to 0   | niskie               |
| bin_3     | 0           | dominująca masa      |
| bin_4     | 0 to 0.5    | małe dodatnie        |
| bin_5     | 0.5 to 1    | średnie dodatnie     |
| bin_6     | 1 to 2      | umiarkowane dodatnie |
| bin_7     | 2 to 5      | wysokie dodatnie     |
| bin_8     | 5 to 10     | bardzo wysokie       |
| bin_9     | 10+         | ekstremalne wartości |

In [63]:
label_bin_distribution = merged_df.groupby('score_bin')['label'].value_counts()
print("Label distribution by bin:")
label_bin_distribution

Label distribution by bin:


score_bin  label  
-0.5       NO           90
           INVALID      37
           YES           1
-1.0       NO          118
           INVALID      43
           YES           2
0.0        NO         2438
           INVALID    1549
           YES          31
0.5        NO           66
           INVALID      32
           YES           3
1.0        NO          136
           INVALID     100
           YES           8
1.5-2.0    NO           47
           INVALID      34
           YES           7
10.5+      NO            6
           INVALID       4
           YES           2
2.5-5.0    INVALID      27
           NO           24
           YES           9
5.5-10.0   NO            7
           YES           5
           INVALID       3
Name: count, dtype: int64

In [65]:
df_inv_to_remove = pd.read_csv("../data_all/above_value_minus1/invalid_but_responding.csv")
df_inv_to_remove

,uid
0,b90dd90742072d7f1e0bb832810a1a86
1,e053d6a83b40a215b70b536499a1fbb6
2,8666387e4fcca13ca92ac62413398a15
3,d63bf2d22ecd4894a8c0fb52235bc267
4,86bebb95232151561968e98c9e52055f
...,...
117,b10b80665c1b63b98d7057a3cef91ca8
118,c65823d5d604ef94f31ad2c1d6bc96af
119,3413a9ed01919372d2d04ade590f378c
120,821318b0521fca2458a7838b3ddcf092


In [66]:
merged_df.shape

(4829, 8)

In [67]:
uids_to_remove = df_inv_to_remove['uid'].tolist()
merged_df = merged_df[~merged_df['uid'].isin(uids_to_remove)]

In [68]:
merged_df.shape

(4724, 8)

In [69]:
df_to_check = merged_df[['uid', 'url', 'label']]
df_to_check.head()

,uid,url,label
0,47d936b9a2b73729f100922aaea4754b,https://assets.weather-forecast.com/maps/stati...,NO
1,eb7ed790974fe9549d8ec73ef1b5a40f,http://actualidadhumanitaria.com/wp-content/up...,NO
2,026ad7cea92db1bc23e34058544ee81b,https://i.hizliresim.com/6yAr9v.jpg,INVALID
3,2deac59fba180138c78559a999b9654a,https://static1.millenium.org/article_old/imag...,NO
4,e72246916a1d6af335ef2d588e965282,https://volcanodiscovery.de/maps/quakemap-6127...,NO


In [71]:
# distribution df_to_check without INVALID
valid_label_distribution = df_to_check[df_to_check['label'] != 'INVALID']['label'].value_counts()
print("Valid label distribution (excluding INVALID):")
print(valid_label_distribution)

Valid label distribution (excluding INVALID):
label
NO     2932
YES      68
Name: count, dtype: int64


In [72]:
df_to_check.to_parquet("../data_all/above_value_minus1/annotations_to_check.parquet", index=False)